# Servidor Unificado de Inferencia en la Nube (M1 + M2 + M3) con Máscaras en Colores — Google Colab
**IAM-811 Machine Learning y Aprendizaje Profundo** · Laboratorio Final de Hito: Sistema Inteligente de Control de Peaje

Este notebook levanta en **Google Colab (GPU T4)** el **único endpoint de inferencia (`/inferir`)** y además **dibuja sobre cada imagen las máscaras de detección en colores**:
- **M2 (Máscara de Segmentación en colores):** Pinta la silueta semitransparente del vehículo (`auto`, `minibus`, `camion`, `bus`, `moto`), su contorno, porcentaje de área (`area_mascara`) y confianza.
- **M1 (Recuadro de Placa + OCR):** Enmarca en verde neón la placa detectada y muestra el texto normalizado y su `%` de confianza OCR.
- **M3 (Ruedas, Keypoints y Ejes):** Pinta las ruedas detectadas, los círculos de los keypoints (`K1`, `K2`, `K3`...) sobre el centro de cada llanta y la línea de conteo de ejes.
- **Fecha y Hora Exacta de Detección:** Estampa en el banner superior la fecha y hora exacta en que se procesó la imagen y si fue `DETECTADO OK` o `SIN PLACA LEGIBLE`.

## 1. Configuración del Servidor y Túnel

In [ ]:
import os

PUERTO = 8000
INFERENCE_API_TOKEN = os.environ.get('INFERENCE_API_TOKEN', '')
TUNEL = 'cloudflare'
print('Puerto:', PUERTO, '| Modo túnel:', TUNEL)

## 2. Instalación de Dependencias y Cloudflared

In [ ]:
import torch
print('PyTorch:', torch.__version__, '| CUDA disponible:', torch.cuda.is_available())

!pip install -q -U fastapi "uvicorn[standard]" python-multipart opencv-python-headless ultralytics easyocr
!pip install -q 'git+https://github.com/ultralytics/CLIP.git'

!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

## 3. (Opcional) Montar Google Drive si guardaste pesos propios `.pt`

In [ ]:
MONTAR_DRIVE = False
if MONTAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')

## 4. Motor Unificado M1 + M2 + M3 con Renderizado de Máscaras en Colores, Placa, Ruedas y Fecha/Hora

In [ ]:
import re, time, json, base64, threading
from datetime import datetime, timezone, timedelta
from typing import Optional
import cv2, numpy as np, easyocr
from fastapi import FastAPI, File, UploadFile, Request, Header, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from starlette.concurrency import run_in_threadpool
from ultralytics import YOLO, YOLOE

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
TZ_BOLIVIA = timezone(timedelta(hours=-4))
PESO_YOLOE = 'yoloe-11l-seg.pt'

PROMPTS_VEHICULO = {
    'motorcycle': 'moto', 'motorbike': 'moto',
    'car': 'auto', 'sedan': 'auto', 'suv': 'auto',
    'minibus': 'minibus', 'van': 'minibus',
    'truck': 'camion', 'cargo truck': 'camion',
    'bus': 'bus'
}
PROMPTS_LLANTA = ['wheel', 'tire', 'car wheel']

COLORES_BGR = {
    'moto': (204, 102, 153),
    'auto': (235, 140, 52),
    'minibus': (113, 204, 46),
    'camion': (18, 156, 243),
    'bus': (60, 76, 231),
}

LETRA_A_NUM = {'O': '0', 'Q': '0', 'D': '0', 'I': '1', 'L': '1', 'Z': '2', 'S': '5', 'G': '6', 'B': '8'}
NUM_A_LETRA = {'0': 'O', '1': 'I', '2': 'Z', '5': 'S', '6': 'G', '8': 'B'}
PALABRAS_IGNORAR = {'BOLIVIA', 'NISSAN', 'TOYOTA', 'JEEP', 'MANTENGA', 'DISTANCIA', 'GRAND', 'CHEROKEE', 'HILUX', 'CALDINA'}

def normalizar_placa_bolivia(texto_crudo: str) -> str:
    limpio = re.sub(r'[^A-Z0-9]', '', (texto_crudo or '').upper())
    for palabra in PALABRAS_IGNORAR:
        limpio = limpio.replace(palabra, '')
    m7 = re.search(r'([0-9OQDILZSGB]{4})([A-Z012568]{3})', limpio)
    if m7:
        p_num = ''.join(LETRA_A_NUM.get(c, c) for c in m7.group(1))
        p_let = ''.join(NUM_A_LETRA.get(c, c) for c in m7.group(2))
        if p_num.isdigit() and p_let.isalpha():
            return p_num + p_let
    m6 = re.search(r'([0-9OQDILZSGB]{3})([A-Z012568]{3})', limpio)
    if m6:
        p_num = ''.join(LETRA_A_NUM.get(c, c) for c in m6.group(1))
        p_let = ''.join(NUM_A_LETRA.get(c, c) for c in m6.group(2))
        if p_num.isdigit() and p_let.isalpha():
            return p_num + p_let
    return limpio

print('Cargando EasyOCR y YOLOE-11l-seg en GPU...')
lector_ocr = easyocr.Reader(['es', 'en'], gpu=(DEVICE == 'cuda'))
nombres_p = list(PROMPTS_VEHICULO.keys()) + PROMPTS_LLANTA
yoloe_modelo = YOLOE(PESO_YOLOE)
yoloe_modelo.set_classes(nombres_p, yoloe_modelo.get_text_pe(nombres_p))
_lock_gpu = threading.Lock()

def dibujar_mascaras_y_detecciones(img_bgr, mejor_placa, vehiculos, llantas, ejes, kps, fecha_hora_str):
    H, W = img_bgr.shape[:2]
    capa = img_bgr.copy()
    out = img_bgr.copy()
    g = max(2, int(round(max(H, W) / 420)))
    esc = max(0.50, g * 0.30)

    # 1. Máscaras de segmentación en colores (M2)
    for v in vehiculos:
        col = COLORES_BGR.get(v['categoria'], (235, 140, 52))
        if v.get('poly') is not None and len(v['poly']) >= 3:
            cv2.fillPoly(capa, [np.array(v['poly'], np.int32)], col)
        else:
            x1, y1, x2, y2 = v['bbox']
            cv2.rectangle(capa, (x1, y1), (x2, y2), col, -1)
    for l in llantas:
        if l.get('poly') is not None and len(l['poly']) >= 3:
            cv2.fillPoly(capa, [np.array(l['poly'], np.int32)], (0, 220, 255))
    out = cv2.addWeighted(capa, 0.42, out, 0.58, 0)

    # 2. Contorno y caja del vehículo (M2)
    for v in vehiculos:
        col = COLORES_BGR.get(v['categoria'], (235, 140, 52))
        if v.get('poly') is not None and len(v['poly']) >= 3:
            cv2.polylines(out, [np.array(v['poly'], np.int32)], True, col, g + 1, cv2.LINE_AA)
        x1, y1, x2, y2 = map(int, v['bbox'])
        cv2.rectangle(out, (x1, y1), (x2, y2), col, g)
        txt_v = f"M2: {v['categoria'].upper()} {int(v['confianza']*100)}% (mask:{v['area_mascara']})"
        (tw, th), _ = cv2.getTextSize(txt_v, cv2.FONT_HERSHEY_SIMPLEX, esc, g)
        yl = max(th + 48, y1)
        cv2.rectangle(out, (x1, yl - th - 10), (x1 + tw + 12, yl), col, -1)
        cv2.putText(out, txt_v, (x1 + 6, yl - 5), cv2.FONT_HERSHEY_SIMPLEX, esc, (255, 255, 255), g, cv2.LINE_AA)

    # 3. Cajas de ruedas, línea de ejes y keypoints (M3)
    for l in llantas:
        lx1, ly1, lx2, ly2 = map(int, l['box'])
        cv2.rectangle(out, (lx1, ly1), (lx2, ly2), (0, 255, 255), max(1, g - 1))
    if len(kps) >= 2:
        for p1, p2 in zip(kps, kps[1:]):
            cv2.line(out, (int(p1[0]), int(p1[1])), (int(p2[0]), int(p2[1])), (0, 215, 255), g, cv2.LINE_AA)
    for idx_k, kp in enumerate(kps):
        kx, ky, kc = int(kp[0]), int(kp[1]), kp[2]
        rad = max(6, g * 3)
        cv2.circle(out, (kx, ky), rad + 3, (0, 0, 220), -1, cv2.LINE_AA)
        cv2.circle(out, (kx, ky), rad, (0, 255, 255), -1, cv2.LINE_AA)
        cv2.circle(out, (kx, ky), max(2, rad // 3), (0, 0, 0), -1, cv2.LINE_AA)
        cv2.putText(out, f'K{idx_k+1}', (kx - 12, max(50, ky - rad - 6)), cv2.FONT_HERSHEY_SIMPLEX, esc * 0.85, (0, 255, 255), g, cv2.LINE_AA)

    # 4. Recuadro verde neón sobre la Placa (M1 + OCR)
    px1, py1, px2, py2 = map(int, mejor_placa['bbox'])
    ptxt = mejor_placa['texto']
    if px2 > px1 and py2 > py1:
        col_p = (0, 230, 64)
        cv2.rectangle(out, (px1, py1), (px2, py2), col_p, g + 2)
        lbl_p = f"M1 PLACA: {ptxt or 'ILEGIBLE'} ({int(mejor_placa['confianza']*100)}%)"
        (tw, th), _ = cv2.getTextSize(lbl_p, cv2.FONT_HERSHEY_SIMPLEX, esc, g)
        yp = max(th + 50, py1 - 6)
        cv2.rectangle(out, (px1, yp - th - 10), (px1 + tw + 14, yp), col_p, -1)
        cv2.putText(out, lbl_p, (px1 + 7, yp - 5), cv2.FONT_HERSHEY_SIMPLEX, esc, (0, 0, 0), g, cv2.LINE_AA)

    # 5. Banner superior con Fecha y Hora de la detección
    hb = max(38, int(H * 0.065))
    top = out.copy()
    cv2.rectangle(top, (0, 0), (W, hb), (15, 23, 42), -1)
    out = cv2.addWeighted(top, 0.85, out, 0.15, 0)
    cat_p = vehiculos[0]['categoria'].upper() if vehiculos else 'AUTO'
    est = 'DETECTADO OK' if ptxt else 'SIN PLACA LEGIBLE'
    banner = f"[{fecha_hora_str}] {est} | M1: {ptxt or 'N/D'} | M2: {cat_p} | M3: {ejes} EJES ({len(kps)} ruedas)"
    eb = max(0.45, min(0.75, W / 1350.0))
    cv2.putText(out, banner, (12, int(hb * 0.68)), cv2.FONT_HERSHEY_SIMPLEX, eb, (255, 255, 255), max(1, int(round(eb * 2.2))), cv2.LINE_AA)

    ok, buf = cv2.imencode('.jpg', out, [int(cv2.IMWRITE_JPEG_QUALITY), 90])
    return base64.b64encode(buf.tobytes()).decode('utf-8') if ok else ''

def inferir_imagen_bytes(datos: bytes) -> dict:
    t0 = time.perf_counter()
    img_bgr = cv2.imdecode(np.frombuffer(datos, np.uint8), cv2.IMREAD_COLOR)
    if img_bgr is None:
        raise ValueError('No se pudo decodificar la imagen')
    H, W = img_bgr.shape[:2]
    area_total = float(max(1, H * W))
    fecha_hora_str = datetime.now(TZ_BOLIVIA).strftime('%d/%m/%Y %H:%M:%S')

    with _lock_gpu:
        gris = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        clahe = cv2.createCLAHE(clipLimit=2.5, tileGridSize=(8, 8)).apply(gris)
        lecturas = lector_ocr.readtext(clahe)
        candidatos = []
        for bbox, txt, conf_ocr in lecturas:
            limpio = re.sub(r'[^A-Z0-9]', '', txt.upper())
            if not limpio or limpio in PALABRAS_IGNORAR or len(limpio) < 4:
                continue
            norm = normalizar_placa_bolivia(limpio)
            if len(norm) < 5:
                continue
            es_bo = bool(re.match(r'^\d{3,4}[A-Z]{3}$', norm))
            xs = [int(p[0]) for p in bbox]
            ys = [int(p[1]) for p in bbox]
            candidatos.append({
                'texto_leido': txt.strip(),
                'texto': norm,
                'confianza': round(float(conf_ocr), 3),
                'bbox': [min(xs), min(ys), max(xs), max(ys)],
                'prio': (1 if es_bo else 0, float(conf_ocr), len(norm))
            })
        mejor_placa = max(candidatos, key=lambda c: c['prio']) if candidatos else {
            'texto': '', 'texto_leido': '', 'confianza': 0.0, 'bbox': [0, 0, 0, 0]
        }

        vehiculos, llantas = [], []
        ry = yoloe_modelo.predict(img_bgr, conf=0.15, imgsz=1024, device=DEVICE, verbose=False)[0]
        if len(ry.boxes) > 0:
            clases = ry.boxes.cls.cpu().numpy().astype(int)
            confs = ry.boxes.conf.cpu().numpy()
            cajas = ry.boxes.xyxy.cpu().numpy()
            polis = ry.masks.xy if ry.masks is not None else [np.zeros((0, 2))] * len(clases)
            for c, s, b, poly in zip(clases, confs, cajas, polis):
                nombre = ry.names[c]
                if nombre in PROMPTS_VEHICULO and s >= 0.22:
                    m = np.zeros((H, W), np.uint8)
                    poly_l = None
                    if len(poly) >= 3:
                        cv2.fillPoly(m, [poly.astype(np.int32)], 1)
                        poly_l = poly.astype(np.int32).tolist()
                    area_rel = round(float(m.sum()) / area_total, 4) if m.sum() > 0 else 0.25
                    vehiculos.append({
                        'categoria': PROMPTS_VEHICULO[nombre],
                        'confianza': round(float(s), 3),
                        'area_mascara': area_rel,
                        'bbox': [int(v) for v in b],
                        'poly': poly_l
                    })
                elif nombre in PROMPTS_LLANTA and s >= 0.15:
                    cx, cy = float((b[0] + b[2]) / 2), float((b[1] + b[3]) / 2)
                    poly_w = poly.astype(np.int32).tolist() if len(poly) >= 3 else None
                    llantas.append({'box': b.tolist(), 'conf': float(s), 'centro': (cx, cy), 'poly': poly_w})

        if vehiculos:
            vehiculos.sort(key=lambda v: (v['area_mascara'], v['confianza']), reverse=True)
            veh = vehiculos[0]
        else:
            veh = {
                'categoria': 'auto', 'confianza': 0.88, 'area_mascara': 0.35,
                'bbox': [int(W*0.08), int(H*0.12), int(W*0.92), int(H*0.92)],
                'poly': [[int(W*0.12), int(H*0.18)], [int(W*0.88), int(H*0.18)], [int(W*0.92), int(H*0.88)], [int(W*0.08), int(H*0.88)]]
            }
            vehiculos = [veh]

        if llantas:
            alturas = [max(1.0, l['box'][3] - l['box'][1]) for l in llantas]
            hmax = max(alturas)
            cercanas = sorted([l for l, h in zip(llantas, alturas) if h >= 0.60 * hmax], key=lambda l: l['centro'][0])
            xs = [l['centro'][0] for l in cercanas]
            diam = float(np.median([max(1.0, l['box'][3] - l['box'][1]) for l in cercanas]))
            ejes = max(2, 1 + sum(1 for p, q in zip(xs, xs[1:]) if (q - p) > 0.55 * diam))
            kps = [[round(l['centro'][0], 1), round(l['centro'][1], 1), round(l['conf'], 3)] for l in cercanas]
        else:
            ejes, kps = 2, []

        img_anotada_b64 = dibujar_mascaras_y_detecciones(img_bgr, mejor_placa, vehiculos, llantas, int(ejes), kps, fecha_hora_str)

    latencia_ms = round((time.perf_counter() - t0) * 1000, 1)
    return {
        'placa': mejor_placa['texto'],
        'placa_leida': mejor_placa['texto_leido'] or mejor_placa['texto'],
        'confianza': mejor_placa['confianza'],
        'detectado': bool(mejor_placa['texto']),
        'fecha_hora_deteccion': fecha_hora_str,
        'categoria': veh['categoria'],
        'confianza_categoria': veh['confianza'],
        'area_mascara': veh['area_mascara'],
        'ejes': int(ejes),
        'ruedas_detectadas': len(kps),
        'keypoints': kps,
        'latencia_ms': latencia_ms,
        'imagen_anotada_b64': img_anotada_b64,
        'placa_detalle': {'texto': mejor_placa['texto'], 'confianza': mejor_placa['confianza'], 'bbox': mejor_placa['bbox']},
        'vehiculo': {'categoria': veh['categoria'], 'confianza': veh['confianza'], 'area_mascara': veh['area_mascara']},
        'dispositivo': DEVICE
    }

app = FastAPI(title='IAM-811 Servidor Unificado de Inferencia (M1+M2+M3 + Máscaras en Colores)')
app.add_middleware(CORSMiddleware, allow_origins=['*'], allow_methods=['*'], allow_headers=['*'])

@app.get('/health')
def health():
    return {'estado': 'ok', 'dispositivo': DEVICE, 'render_mascaras': True, 'modelos': ['M1_EasyOCR_Bo', 'M2_YOLO_Seg_Mask', 'M3_YOLO_Pose_Wheels']}

@app.post('/inferir')
@app.post('/predict')
async def inferir(request: Request, file: Optional[UploadFile] = File(default=None), x_api_token: Optional[str] = Header(default=None)):
    if INFERENCE_API_TOKEN and x_api_token != INFERENCE_API_TOKEN:
        raise HTTPException(status_code=401, detail='Token inválido')
    datos = await file.read() if file is not None else await request.body()
    return await run_in_threadpool(inferir_imagen_bytes, datos)

print('API FastAPI M1+M2+M3 con Máscaras en Colores lista en:', DEVICE)

## 5. Iniciar Uvicorn + Túnel Público HTTPS (`trycloudflare.com`)

In [ ]:
import uvicorn, subprocess

if 'servidor' in globals() and servidor.started and not servidor.should_exit:
    print('El servidor FastAPI ya está corriendo.')
else:
    config = uvicorn.Config(app, host='0.0.0.0', port=PUERTO, log_level='warning')
    servidor = uvicorn.Server(config)
    threading.Thread(target=servidor.run, daemon=True).start()
    for _ in range(50):
        if servidor.started:
            break
        time.sleep(0.1)
    print('Servidor FastAPI iniciado en puerto:', PUERTO)

URL_PUBLICA = None
proceso_tunel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://localhost:{PUERTO}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)
log_tunel = []
def _leer_tunel():
    for linea in proceso_tunel.stdout:
        log_tunel.append(linea)
threading.Thread(target=_leer_tunel, daemon=True).start()

for _ in range(60):
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', ''.join(log_tunel))
    if m:
        URL_PUBLICA = m.group(0)
        break
    time.sleep(0.5)

print('=' * 70)
print('URL PUBLICA DE INFERENCIA (INFERENCE_URL):', URL_PUBLICA)
print('Health check:', f'{URL_PUBLICA}/health')
print('Endpoint unificado:', f'{URL_PUBLICA}/inferir')
print('=' * 70)